In [13]:
import torch
import torch.nn as nn
import torchinfo
from model.mini_cnn import MiniCNN
from result.repondeur import prediction_to_csv
import torchvision
import numpy as np
import csv
from tqdm import tqdm
from model.loader import load_dataset, load_test_loader

In [14]:
# Check for GPU availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✓ Using GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("✓ Using Apple Silicon GPU (MPS)")
else:
    device = torch.device("cpu")
    print("⚠ Using CPU - training will be slow!")

⚠ Using CPU - training will be slow!


In [15]:
def train_crossEntropyLoss(train_set, model, epochs):
    """
    Entraine un model à partir d'un train_set donné. Retourne les metrics de l'entrainement
    """

    # Parameters
    model.train()

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=0.1,
        momentum=0.9,
        weight_decay=5e-4
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=200
    )


    accuracies = np.zeros(epochs)
    losses = np.zeros(epochs)

    for epoch in range(epochs):
        total_loss = 0.0
        correct = 0
        total = 0

        pbar = tqdm(
            train_set,
            desc=f"Model training | Epoch {epoch+1}/{epochs}",
            leave=True
        )

        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)

            #prediction
            logits = model(images)
            loss = criterion(logits, labels)

            #propagation du gradient
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Calcul des metriques
            total_loss += loss.item()
            
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            pbar.set_postfix({
                "batch_loss": f"{loss.item():.4f}",
                "accuracy": f"{correct/total*100:.2f}%"
            })
        accuracies[epoch] = correct/total
        losses[epoch] = total_loss

        scheduler.step()
        print(accuracies)
        print(losses)
    return accuracies, losses

In [16]:
from torch.utils.data import DataLoader
from torchvision import transforms
from utils.CustomDataset import CustomDataset
from utils.CustomSkibidiDataset import CustomSkibidiDataset

from torchvision.transforms import v2

In [17]:
# 1. Define Image Transforms
# HUUUUGOOOOOOOO
# transform = transforms.Compose([
#     transforms.Resize((224, 224)),
#     torchvision.transforms.functional.to_dtype(float16, scale=True),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])

transform = v2.Compose([
    #v2.ToImage(),                  # Convert to Tensor (if it's PIL)
    v2.Resize((224, 224)),
    v2.ToDtype(torch.float32, scale=True), # Scales to float16 [0, 1] # TODO
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


# 2. Instantiate your Dataset
"""train_dataset = CustomDataset(
    img_dir='data/',
    annotations_file='data/train.csv',
    transform=transform, 
    target_transform=None # ???
)"""

train_dataset = CustomSkibidiDataset(
    img_dir='data/treated_train/',
    csv_mapper='data/treated_train.csv',
    transform=transform, 
    target_transform=None # ???
)

Found folders ['Andrena haemorrhoa', 'Andrena vaga', 'Andrena rufula', 'Andrena vicinoides', 'Andrena carantonica', 'Andrena hattorfiana', 'Andrena barbilabris', 'Andrena wilkella', 'Andrena mendica', 'Andrena melanochroa', 'Andrena rudbeckiae', 'Andrena aerinifrons', 'Andrena nigroaenea', 'Andrena fulvata', 'Andrena villipes', 'Andrena orbitalis', 'Andrena fulva', 'Andrena convallaria', 'Andrena prodigiosa', 'Andrena lineolata', 'Andrena crawfordi', 'Andrena perplexa', 'Andrena cineraria', 'Andrena florivaga', 'Andrena pinguis', 'Andrena banksi', 'Andrena leucophaea', 'Andrena fortipunctata', 'Andrena mediovittata', 'Andrena hesperia', 'Andrena costillensis', 'Andrena limbata', 'Andrena irana', 'Andrena vulpecula', 'Andrena clarkella', 'Andrena nitida', 'Andrena angustitarsata', 'Andrena ventralis', 'Andrena dorsata', 'Andrena bicolor', 'Andrena chengtehensis', 'Andrena plana', 'Andrena denticulata', 'Andrena afrensis', 'Andrena discors', 'Andrena perimelas', 'Andrena aliciae', 'Andre

In [18]:

epochs = 20
model = MiniCNN()
model.to(device)
train_set = DataLoader(
    dataset=train_dataset,
    batch_size=32,      # Number of images per batch
    shuffle=True,       # Shuffle every epoch to prevent overfitting
    num_workers=4,      # Number of CPU cores for data loading
    pin_memory=True     # Speeds up transfer to GPU
)

accuracies, losses = train_crossEntropyLoss(train_set,model, epochs)
try : 
    np.save("./ACCURACIES.npy", accuracies)
    np.save("./LOSSES.npy", losses)
except :
    pass
torch.save(model.state_dict(), f"model/trained_model/MiniCNN_augmented_data_{epochs}_epochs.pth")

Model training | Epoch 1/20:   0%|          | 0/820 [00:00<?, ?it/s]/home/niels/.sdd_venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Model training | Epoch 1/20: 100%|██████████| 820/820 [08:32<00:00,  1.60it/s, batch_loss=2.7962, accuracy=15.85%]


[0.1584664 0.        0.        0.        0.        0.        0.
 0.        0.        0.        0.        0.        0.        0.
 0.        0.        0.        0.        0.        0.       ]
[2620.53375244    0.            0.            0.            0.
    0.            0.            0.            0.            0.
    0.            0.            0.            0.            0.
    0.            0.            0.            0.            0.        ]


Model training | Epoch 2/20: 100%|██████████| 820/820 [08:29<00:00,  1.61it/s, batch_loss=2.2326, accuracy=31.78%]


[0.1584664  0.31777126 0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.        ]
[2620.53375244 2046.22365034    0.            0.            0.
    0.            0.            0.            0.            0.
    0.            0.            0.            0.            0.
    0.            0.            0.            0.            0.        ]


Model training | Epoch 3/20: 100%|██████████| 820/820 [08:26<00:00,  1.62it/s, batch_loss=1.5020, accuracy=40.74%]


...

[0.1584664  0.31777126 0.40744693 0.48378368 0.53443348 0.57178246
 0.60844544 0.62647205 0.64289798 0.65273067 0.67300583 0.68150463
 0.68356264 0.69461489 0.69716834 0.6990739  0.70166546 0.7133656
 0.         0.        ]
[2620.53375244 2046.22365034 1743.56661808 1507.47172689 1352.97244662
 1225.47453034 1129.32174516 1068.72784662 1027.38349605  983.20883095
  942.14634201  915.41320875  903.51299536  871.49593982  868.43220165
  864.52788416  847.12268594  827.91511655    0.            0.        ]


Model training | Epoch 19/20: 100%|██████████| 820/820 [08:28<00:00,  1.61it/s, batch_loss=0.7761, accuracy=70.71%]


[0.1584664  0.31777126 0.40744693 0.48378368 0.53443348 0.57178246
 0.60844544 0.62647205 0.64289798 0.65273067 0.67300583 0.68150463
 0.68356264 0.69461489 0.69716834 0.6990739  0.70166546 0.7133656
 0.70707725 0.        ]
[2620.53375244 2046.22365034 1743.56661808 1507.47172689 1352.97244662
 1225.47453034 1129.32174516 1068.72784662 1027.38349605  983.20883095
  942.14634201  915.41320875  903.51299536  871.49593982  868.43220165
  864.52788416  847.12268594  827.91511655  832.21525401    0.        ]


Model training | Epoch 20/20: 100%|██████████| 820/820 [08:23<00:00,  1.63it/s, batch_loss=0.6694, accuracy=71.17%]

[0.1584664  0.31777126 0.40744693 0.48378368 0.53443348 0.57178246
 0.60844544 0.62647205 0.64289798 0.65273067 0.67300583 0.68150463
 0.68356264 0.69461489 0.69716834 0.6990739  0.70166546 0.7133656
 0.70707725 0.71172682]
[2620.53375244 2046.22365034 1743.56661808 1507.47172689 1352.97244662
 1225.47453034 1129.32174516 1068.72784662 1027.38349605  983.20883095
  942.14634201  915.41320875  903.51299536  871.49593982  868.43220165
  864.52788416  847.12268594  827.91511655  832.21525401  819.20149347]
